# 04 Feature Engineering Validation

Notebook check output của 3 file feature mới: TF-IDF, Word2Vec, FastText.
Nó sẽ chạy từng pipeline và xác nhận các artifact đã được lưu đúng.

In [2]:
from pathlib import Path
import sys
import importlib.util

root = Path.cwd()
if not (root / 'src' / 'features' / 'tfidf_features.py').exists():
    root = next(
        (parent for parent in root.parents if (parent / 'src' / 'features' / 'tfidf_features.py').exists()),
        None,
    )
    if root is None:
        raise FileNotFoundError(
            'Could not locate repo root containing src/features/tfidf_features.py'
        )

src_dir = root / 'src'
sys.path.insert(0, str(src_dir))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


tfidf_features = load_module('tfidf_features', str(src_dir / 'features' / 'tfidf_features.py'))
word2vec_features = load_module('word2vec_features', str(src_dir / 'features' / 'word2vec_features.py'))
fasttext_features = load_module('fasttext_features', str(src_dir / 'features' / 'fasttext_features.py'))

root, src_dir

(WindowsPath('c:/Users/THINKPAD/Downloads/Customer_sentiment_analysis'),
 WindowsPath('c:/Users/THINKPAD/Downloads/Customer_sentiment_analysis/src'))

In [3]:
output_paths = {}
try:
    output_paths['tfidf'] = tfidf_features.run_tfidf_pipeline()
except Exception as exc:
    print('TF-IDF pipeline failed:', exc)

try:
    output_paths['word2vec'] = word2vec_features.run_word2vec_pipeline()
except Exception as exc:
    print('Word2Vec pipeline failed:', exc)

try:
    output_paths['fasttext'] = fasttext_features.run_fasttext_pipeline()
except Exception as exc:
    print('FastText pipeline failed:', exc)

output_paths

{'tfidf': WindowsPath('C:/Users/THINKPAD/Downloads/Customer_sentiment_analysis/data/processed/tfidf_features.pkl'),
 'word2vec': WindowsPath('C:/Users/THINKPAD/Downloads/Customer_sentiment_analysis/data/processed/word2vec_features.pkl'),
 'fasttext': WindowsPath('C:/Users/THINKPAD/Downloads/Customer_sentiment_analysis/data/processed/fasttext_features.pkl')}

In [4]:
import joblib

artifact_checks = {}
for name, path in output_paths.items():
    path_obj = Path(path) if path is not None else None
    artifact_checks[name] = {
        'path': str(path_obj) if path_obj is not None else None,
        'exists': path_obj.exists() if path_obj is not None else False,
    }
    if path_obj is not None and path_obj.exists():
        artifact = joblib.load(path_obj)
        artifact_checks[name]['artifact_keys'] = list(artifact.keys())
        if 'matrix' in artifact:
            artifact_checks[name]['matrix_shape'] = artifact['matrix'].shape
        if 'doc_embeddings' in artifact:
            artifact_checks[name]['doc_embeddings_shape'] = artifact['doc_embeddings'].shape

artifact_checks

{'tfidf': {'path': 'C:\\Users\\THINKPAD\\Downloads\\Customer_sentiment_analysis\\data\\processed\\tfidf_features.pkl',
  'exists': True,
  'artifact_keys': ['vectorizer', 'matrix', 'ids', 'labels'],
  'matrix_shape': (10947, 7308)},
 'word2vec': {'path': 'C:\\Users\\THINKPAD\\Downloads\\Customer_sentiment_analysis\\data\\processed\\word2vec_features.pkl',
  'exists': True,
  'artifact_keys': ['model_path',
   'doc_embeddings',
   'ids',
   'labels',
   'vector_size'],
  'doc_embeddings_shape': (10947, 100)},
 'fasttext': {'path': 'C:\\Users\\THINKPAD\\Downloads\\Customer_sentiment_analysis\\data\\processed\\fasttext_features.pkl',
  'exists': True,
  'artifact_keys': ['model_path',
   'doc_embeddings',
   'ids',
   'labels',
   'vector_size'],
  'doc_embeddings_shape': (10947, 100)}}